# Mã hóa biến → SMOTE
**Dữ liệu:** `xuly1.csv` (đã làm sạch)  
**Target:** `NHOMNOMOI` (5 nhóm, mất cân bằng nặng)

In [ ]:

#pip install imbalanced-learn scikit-learn pandas numpy

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from imblearn.over_sampling import SMOTE

print('Thư viện load thành công!')

Thư viện load thành công!


---
## 1. Load dữ liệu

In [3]:
df = pd.read_csv('xuly1.csv')
print('Shape:', df.shape)
df.head()

Shape: (93288, 13)


,MJACCTTYPCD,PHUONG THUC CHO VAY,LOAIKH,SEX,BASE_BAL,DUNO_QD,THOIHAN,DESC_TIME,MJACCTTYPDESC,PARENTORGNAME,LAISUAT,MUCDICHVAY,NHOMNOMOI
0,CNS,TRA GOP,1,1,400000000.0,350000000.0,3,Vay ngan han,Vay Tieu dung,KHANH HOA,12.0,1900,1
1,CNS,TRA GOP,1,1,333388030.0,142450650.0,8,Vay ngan han,Vay Tieu dung,DA NANG,18.0,1811,1
2,CNS,TRA GOP,1,1,311014800.0,350000000.0,8,Vay ngan han,Vay Tieu dung,DA NANG,18.0,1870,1
3,CNS,CV TUNG LAN LAI DINH KY,1,1,35600000.0,35600000.0,37,Vay trung han,Vay Tieu dung,KHANH HOA,24.0,1830,5
4,CNS,TRA GOP,1,1,400000000.0,88246180.0,8,Vay ngan han,Vay Tieu dung,DA NANG,18.0,1870,1


In [4]:
print('Kiểu dữ liệu:\n')
print(df.dtypes)
print('\nPhân phối target - NHOMNOMOI:')
print(df['NHOMNOMOI'].value_counts().sort_index())
print('\nTỷ lệ %:')
print((df['NHOMNOMOI'].value_counts(normalize=True).sort_index() * 100).round(2))

Kiểu dữ liệu:

MJACCTTYPCD             object
PHUONG THUC CHO VAY     object
LOAIKH                   int64
SEX                      int64
BASE_BAL               float64
DUNO_QD                float64
THOIHAN                  int64
DESC_TIME               object
MJACCTTYPDESC           object
PARENTORGNAME           object
LAISUAT                float64
MUCDICHVAY               int64
NHOMNOMOI                int64
dtype: object

Phân phối target - NHOMNOMOI:
NHOMNOMOI
1    83413
2     3863
3     1286
4     1337
5     3389
Name: count, dtype: int64

Tỷ lệ %:
NHOMNOMOI
1    89.41
2     4.14
3     1.38
4     1.43
5     3.63
Name: proportion, dtype: float64


---
## 2. Mã hóa biến (Encoding)

| Cột | Loại | Phương pháp |
|---|---|---|
| `MJACCTTYPCD` | Nominal (3 giá trị) | Label Encoding |
| `PHUONG THUC CHO VAY` | Nominal (11 giá trị) | Label Encoding |
| `DESC_TIME` | Ordinal (ngắn < trung < dài) | Ordinal Encoding |
| `MJACCTTYPDESC` | Nominal (3 giá trị) | Label Encoding |
| `PARENTORGNAME` | Nominal (31 giá trị) | Label Encoding |

In [5]:
df_encoded = df.copy()

# ── 2.1 Ordinal Encoding cho DESC_TIME ────────────────────
# Thứ tự có ý nghĩa: ngắn hạn < trung hạn < dài hạn
ordinal_enc = OrdinalEncoder(
    categories=[['Vay ngan han', 'Vay trung han', 'Vay dai han']]
)
df_encoded['DESC_TIME'] = ordinal_enc.fit_transform(
    df_encoded[['DESC_TIME']]
).astype(int)

print('DESC_TIME sau Ordinal Encoding:')
print(df_encoded['DESC_TIME'].value_counts().sort_index())

DESC_TIME sau Ordinal Encoding:
DESC_TIME
0    72978
1    12523
2     7787
Name: count, dtype: int64


In [7]:
# ── 2.2 Label Encoding cho các cột Nominal ─────────────────
label_cols = ['MJACCTTYPCD', 'PHUONG THUC CHO VAY', 'MJACCTTYPDESC', 'PARENTORGNAME']

label_encoders = {}  # Lưu lại encoder để decode sau nếu cần

for col in label_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le
    print(f'{col}: {le.classes_} → {list(range(len(le.classes_)))}')



MJACCTTYPCD: ['0' '1' '2'] → [0, 1, 2]
PHUONG THUC CHO VAY: ['0' '1' '10' '2' '3' '4' '5' '6' '7' '8' '9'] → [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
MJACCTTYPDESC: ['0' '1' '2'] → [0, 1, 2]
PARENTORGNAME: ['0' '1' '10' '11' '12' '13' '14' '15' '16' '17' '18' '19' '2' '20' '21'
 '22' '23' '24' '25' '26' '27' '28' '29' '3' '30' '4' '5' '6' '7' '8' '9'] → [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]


In [8]:
# Kiểm tra kết quả sau mã hóa
print('Kiểu dữ liệu sau mã hóa:\n')
print(df_encoded.dtypes)
print('\nPreview:')
df_encoded.head()

Kiểu dữ liệu sau mã hóa:

MJACCTTYPCD              int64
PHUONG THUC CHO VAY      int64
LOAIKH                   int64
SEX                      int64
BASE_BAL               float64
DUNO_QD                float64
THOIHAN                  int64
DESC_TIME                int64
MJACCTTYPDESC            int64
PARENTORGNAME            int64
LAISUAT                float64
MUCDICHVAY               int64
NHOMNOMOI                int64
dtype: object

Preview:


,MJACCTTYPCD,PHUONG THUC CHO VAY,LOAIKH,SEX,BASE_BAL,DUNO_QD,THOIHAN,DESC_TIME,MJACCTTYPDESC,PARENTORGNAME,LAISUAT,MUCDICHVAY,NHOMNOMOI
0,1,2,1,1,400000000.0,350000000.0,3,0,2,9,12.0,1900,1
1,1,2,1,1,333388030.0,142450650.0,8,0,2,2,18.0,1811,1
2,1,2,1,1,311014800.0,350000000.0,8,0,2,2,18.0,1870,1
3,1,9,1,1,35600000.0,35600000.0,37,1,2,9,24.0,1830,5
4,1,2,1,1,400000000.0,88246180.0,8,0,2,2,18.0,1870,1


---
## 3. Tách X và y → Train/Test Split

In [9]:
X = df_encoded.drop(columns=['NHOMNOMOI'])
y = df_encoded['NHOMNOMOI']

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nFeatures:', X.columns.tolist())

X shape: (93288, 12)
y shape: (93288,)

Features: ['MJACCTTYPCD', 'PHUONG THUC CHO VAY', 'LOAIKH', 'SEX', 'BASE_BAL', 'DUNO_QD', 'THOIHAN', 'DESC_TIME', 'MJACCTTYPDESC', 'PARENTORGNAME', 'LAISUAT', 'MUCDICHVAY']


In [10]:
# Tách train/test trước khi SMOTE để tránh data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y         # Giữ tỷ lệ nhóm giống nhau ở cả train và test
)

print(f'Train: {X_train.shape[0]} mẫu | Test: {X_test.shape[0]} mẫu')
print('\nPhân phối y_train trước SMOTE:')
print(y_train.value_counts().sort_index())

Train: 74630 mẫu | Test: 18658 mẫu

Phân phối y_train trước SMOTE:
NHOMNOMOI
1    66730
2     3090
3     1029
4     1070
5     2711
Name: count, dtype: int64


---
## 4. Áp dụng SMOTE (chỉ trên tập Train)


In [16]:
smote = SMOTE(
    sampling_strategy='not majority',  # Oversample tất cả nhóm thiểu số về bằng nhóm đa số
    k_neighbors=5,
    random_state=42
)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'\nX_train trước SMOTE: {X_train.shape}')
print(f'X_train sau  SMOTE: {X_train_sm.shape}')


X_train trước SMOTE: (74630, 12)
X_train sau  SMOTE: (333650, 12)


In [12]:
# So sánh phân phối trước và sau SMOTE
compare = pd.DataFrame({
    'Trước SMOTE': y_train.value_counts().sort_index(),
    'Sau SMOTE':   pd.Series(y_train_sm).value_counts().sort_index()
})
compare['Số mẫu sinh thêm'] = compare['Sau SMOTE'] - compare['Trước SMOTE']
compare.index.name = 'NHOMNOMOI'
print(compare)
print(f'\nTổng mẫu sinh thêm: {compare["Số mẫu sinh thêm"].sum():,}')

           Trước SMOTE  Sau SMOTE  Số mẫu sinh thêm
NHOMNOMOI                                          
1                66730      66730                 0
2                 3090      66730             63640
3                 1029      66730             65701
4                 1070      66730             65660
5                 2711      66730             64019

Tổng mẫu sinh thêm: 259,020


In [13]:
# Tỷ lệ % sau SMOTE
after_pct = (pd.Series(y_train_sm).value_counts(normalize=True).sort_index() * 100).round(2)
print('Tỷ lệ % các nhóm sau SMOTE:')
print(after_pct)

Tỷ lệ % các nhóm sau SMOTE:
NHOMNOMOI
1    20.0
2    20.0
3    20.0
4    20.0
5    20.0
Name: proportion, dtype: float64


---
## 5. Lưu dữ liệu sau SMOTE

In [15]:
# Lưu train sau SMOTE
df_train_smote = pd.DataFrame(X_train_sm, columns=X.columns)
df_train_smote['NHOMNOMOI'] = y_train_sm.values
df_train_smote.to_csv('train_after_smote.csv', index=False)

# Lưu test (giữ nguyên)
df_test = pd.DataFrame(X_test, columns=X.columns)
df_test['NHOMNOMOI'] = y_test.values
df_test.to_csv('test_set.csv', index=False)


print(f'  train_after_smote.csv  — {df_train_smote.shape[0]:,} mẫu')
print(f'  test_set.csv           — {df_test.shape[0]:,} mẫu')

  train_after_smote.csv  — 333,650 mẫu
  test_set.csv           — 18,658 mẫu


---
## Tóm tắt kết quả

| Bước | Kết quả |
|---|---|
| Dữ liệu gốc | 93,288 mẫu |
| Biến categorical mã hóa | 5 cột (1 Ordinal, 4 Label) |
| Train set | 80% |
| Test set | 20% (giữ nguyên, không SMOTE) |
| Sau SMOTE | Tất cả nhóm cân bằng = nhóm đa số |

